# Target-distribution evidence | Issue #19

This notebook reproduces the Gate 3 target-distribution evidence from the frozen Allstate claims source. It verifies the required target statistics, explains the long right tail, and publishes the approved minimal evidence bundle: one summary CSV and one combined figure.

A `long right tail` means most claim losses are relatively low to moderate. However, a small number of claims have much higher losses, extending far to the right of the distribution. 

All headline statistics remain in original `loss` units. `log1p(loss)` is used only to make the distribution shape easier to see.

## Setup and frozen-source contract

The repository root is discovered dynamically so the notebook works from either the repository root or a nested notebook directory. Before analysis, the source file is checked against the frozen byte size, SHA-256 digest, shape, and ordered schema.

In [ ]:
from hashlib import sha256
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter # this is used to format the y-axis labels in scientific notation
import numpy as np
import pandas as pd

# gets the root of the repository by looking for the data/allstate_claims_data.csv file
def find_repo_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "allstate_claims_data.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/allstate_claims_data.csv from the current directory."
    )


# Computes the SHA256 hash of a file in chunks to avoid loading the entire file into memory.
def file_sha256(path, chunk_size=1024 * 1024):
    digest = sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "allstate_claims_data.csv"
EVIDENCE_DIR = (
    REPO_ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 3"
    / "target_distribution_evidence"
)
SUMMARY_PATH = EVIDENCE_DIR / "target_distribution_summary.csv"
FIGURE_PATH = EVIDENCE_DIR / "target_distribution_views.png"

EXPECTED_FILE_SIZE = 70_025_339
EXPECTED_SHA256 = "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
EXPECTED_SHAPE = (188_318, 132)
EXPECTED_HEADER = (
    ["id"]
    + [f"cat{i}" for i in range(1, 117)]
    + [f"cont{i}" for i in range(1, 15)]
    + ["loss"]
)

EVIDENCE_DIR.mkdir(parents=True, exist_ok=True) # create evidence directory if it doesn't exist

In [ ]:
# check the source data file for expected size, hash, shape, and schema

observed_file_size = DATA_PATH.stat().st_size
observed_sha256 = file_sha256(DATA_PATH)

assert observed_file_size == EXPECTED_FILE_SIZE, (
    f"Unexpected source size: {observed_file_size:,} bytes"
)
assert observed_sha256 == EXPECTED_SHA256, (
    f"Unexpected source SHA-256: {observed_sha256}"
)

df = pd.read_csv(DATA_PATH)
assert df.shape == EXPECTED_SHAPE, f"Unexpected source shape: {df.shape}"
assert df.columns.tolist() == EXPECTED_HEADER, "Unexpected ordered source schema"

# observe the target variable and check for missing, non-finite, or non-positive values
loss = pd.to_numeric(df["loss"], errors="raise")
assert loss.notna().all(), "loss contains missing values"
assert np.isfinite(loss.to_numpy()).all(), "loss contains non-finite values"
assert loss.gt(0).all(), "loss must be strictly positive"

print(f"Source: {DATA_PATH.relative_to(REPO_ROOT)}")
print(f"SHA-256: {observed_sha256}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Target validity: complete, finite, and strictly positive")

Source: data/allstate_claims_data.csv
SHA-256: 74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03
Shape: 188,318 rows x 132 columns
Target validity: complete, finite, and strictly positive


## Original-unit summary and severity-tail evidence

High-loss observations use the previously established 1.5 × IQR rule. Values above the upper fence are reported as part of the observed severity distribution. This statistical flag does not establish that a claim is invalid and does not authorize deletion, clipping, or replacement.

`Upper fence` is the statistical threshold for unsually high target values using the 1.5xIQR rule.

In [ ]:
quantile_values = loss.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
q1_loss = float(quantile_values.loc[0.25])
q3_loss = float(quantile_values.loc[0.75])
iqr_loss = q3_loss - q1_loss
upper_iqr_fence = q3_loss + 1.5 * iqr_loss
high_loss_mask = loss > upper_iqr_fence
p99_loss = float(quantile_values.loc[0.99])
raw_hist_excluded_mask = loss > p99_loss

target_distribution_summary = pd.DataFrame(
    [
        {
            "source_rows": int(len(df)),
            "count_loss": int(loss.count()),
            "mean_loss": float(loss.mean()),
            "median_loss": float(loss.median()),
            "std_loss": float(loss.std()),
            "min_loss": float(loss.min()),
            "p25_loss": q1_loss,
            "p50_loss": float(quantile_values.loc[0.50]),
            "p75_loss": q3_loss,
            "p90_loss": float(quantile_values.loc[0.90]),
            "p95_loss": float(quantile_values.loc[0.95]),
            "p99_loss": p99_loss,
            "max_loss": float(loss.max()),
            "raw_skewness": float(loss.skew()),
            "iqr_loss": iqr_loss,
            "upper_iqr_fence": upper_iqr_fence,
            "high_loss_count": int(high_loss_mask.sum()),
            "high_loss_share": float(high_loss_mask.mean()),
            "raw_hist_excluded_count": int(raw_hist_excluded_mask.sum()),
            "raw_hist_excluded_share": float(raw_hist_excluded_mask.mean()),
        }
    ]
)

target_distribution_summary.to_csv(SUMMARY_PATH, index=False)
target_distribution_summary.T.rename(columns={0: "observed_value"})

,observed_value
source_rows,188318.000000
count_loss,188318.000000
mean_loss,3037.337686
median_loss,2115.570000
std_loss,2904.086186
min_loss,0.670000
p25_loss,1204.460000
p50_loss,2115.570000
p75_loss,3864.045000
p90_loss,6401.743000


## Required-value verification

The assertions below allow only the documented rounding differences. They also verify the established high-loss threshold and count and read the published CSV back from disk.

In [ ]:
summary_row = target_distribution_summary.iloc[0] # get the first row of the summary DataFrame for assertions

assert round(summary_row["mean_loss"], 2) == 3_037.34
assert round(summary_row["median_loss"], 2) == 2_115.57
assert round(summary_row["p95_loss"], 2) == 8_508.54
assert round(summary_row["max_loss"], 2) == 121_012.25
assert round(summary_row["raw_skewness"], 3) == 3.795
assert round(summary_row["upper_iqr_fence"], 2) == 7_853.42
assert summary_row["high_loss_count"] == 11_554
assert round(summary_row["high_loss_share"] * 100, 3) == 6.135

published_summary = pd.read_csv(SUMMARY_PATH)
assert published_summary.shape == (1, len(target_distribution_summary.columns))
assert published_summary.columns.tolist() == target_distribution_summary.columns.tolist()
np.testing.assert_allclose( # to_numpy used to convert the DataFrame row to a NumPy array for comparison
    published_summary.iloc[0].to_numpy(dtype=float),
    target_distribution_summary.iloc[0].to_numpy(dtype=float),
    rtol=0,
    atol=1e-12,
)

print("Required Gate 3 values reproduced within documented rounding.")
print(
    f"High-loss tail: {int(summary_row['high_loss_count']):,} rows "
    f"({summary_row['high_loss_share']:.3%}) above ${summary_row['upper_iqr_fence']:,.2f}."
)

Required Gate 3 values reproduced within documented rounding.
High-loss tail: 11,554 rows (6.135%) above $7,853.42.


## Raw and transformed target views

The raw histogram answers where claim amounts fall in original dollars; its x-axis stops at the 99th percentile so the body remains readable, while the annotation reports the excluded share. The full-range raw boxplot keeps the extreme upper tail visible. The `log1p(loss)` histogram compresses large values to reveal the distribution's overall shape, but it does not replace raw-dollar reporting or future MAE calculation.

In [5]:
plot_color = "#2F6B8A"
accent_color = "#D97706"
figure, axes = plt.subplots(1, 3, figsize=(18, 5.5))

axes[0].hist(loss, bins=100, range=(0, p99_loss), color=plot_color, edgecolor="white", linewidth=0.25)
axes[0].set_title("Raw loss through the 99th percentile")
axes[0].set_xlabel("Loss (original units)")
axes[0].set_ylabel("Claims")
axes[0].xaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))
axes[0].text(
    0.98,
    0.95,
    f"Display limit: p99 = ${p99_loss:,.2f}\n"
    f"Above limit: {int(raw_hist_excluded_mask.sum()):,} "
    f"({raw_hist_excluded_mask.mean():.3%})",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.9},
)

axes[1].hist(np.log1p(loss), bins=100, color=accent_color, edgecolor="white", linewidth=0.25)
axes[1].set_title("Full distribution on log1p scale")
axes[1].set_xlabel("log1p(loss), visualization only")
axes[1].set_ylabel("Claims")

axes[2].boxplot(
    loss,
    orientation="vertical",
    patch_artist=True,
    boxprops={"facecolor": plot_color, "alpha": 0.65},
    medianprops={"color": "white", "linewidth": 1.5},
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.25},
)
axes[2].set_title("Raw loss boxplot, full range")
axes[2].set_xticks([1], ["loss"])
axes[2].set_ylabel("Loss (original units)")
axes[2].yaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))

figure.suptitle("Allstate claim-severity target distribution", fontsize=15, y=1.02)
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

## Interpretation and handling decision

The mean loss ($3,037.34) is substantially above the median ($2,115.57), raw skewness is positive (3.795), and the maximum ($121,012.25) is far above the 95th percentile ($8,508.54). Together, these results show a long right tail: most claims are concentrated at lower loss amounts, while a relatively small group of severe claims extends far into higher dollar values.

The raw views preserve business magnitude and show that the upper tail is financially large. The transformed view answers a different question: it makes the shape and concentration of the full distribution readable without changing the underlying records. Therefore, all headline results and future MAE evaluation remain in original `loss` units.

There are 11,554 observations (6.135%) above the $7,853.42 upper IQR fence. They are retained as severity-distribution evidence, not classified as data errors. The frozen source contains no missing, non-finite, or non-positive target values, and this analysis has no independent evidence that any high-loss row is invalid. Any future validity concern must be entered in the anomaly register while preserving the original row until it is resolved.

## Published-artifact verification

In [6]:
assert SUMMARY_PATH.is_file() and SUMMARY_PATH.stat().st_size > 0
assert FIGURE_PATH.is_file() and FIGURE_PATH.stat().st_size > 0

published_image = plt.imread(FIGURE_PATH)
assert published_image.ndim in (2, 3)
assert published_image.shape[0] >= 500 and published_image.shape[1] >= 1_500

published_files = sorted(path.name for path in EVIDENCE_DIR.iterdir() if path.is_file())
expected_files = ["target_distribution_summary.csv", "target_distribution_views.png"]
assert published_files == expected_files, f"Unexpected evidence files: {published_files}"

print("Issue #19 target-distribution evidence verified.")
for path in (SUMMARY_PATH, FIGURE_PATH):
    print(f"- {path.relative_to(REPO_ROOT)} ({path.stat().st_size:,} bytes)")

Issue #19 target-distribution evidence verified.
- notebooks/final-deliverables/September/Gate 3/target_distribution_evidence/target_distribution_summary.csv (510 bytes)
- notebooks/final-deliverables/September/Gate 3/target_distribution_evidence/target_distribution_views.png (103,335 bytes)
